# Prova Final - TransformAI Global Shapers POA

**Avaliacao automatica usando AgentOS + Groq (LLama 3.3)**

Esta prova avalia os conhecimentos adquiridos nas 5 atividades:
1. Tokens
2. Prompt Engineering
3. Temperatura e Criatividade
4. Alucinacoes
5. Etica e Vies

---

**Como funciona:**
1. Inicie o servidor AgentOS: `fastapi dev prova_agent.py`
2. Responda cada questao na celula indicada
3. Execute a celula de avaliacao para receber feedback do agente
4. No final, execute a celula de resultado para ver sua nota total

**Agentes disponiveis:**
- **Professor Avaliador** - Avalia suas respostas
- **Tutor IA** - Tira duvidas (sem dar respostas da prova)

---
> **Tempo estimado:** 30-45 minutos

## Setup - Conexao com AgentOS

**Antes de comecar:**
1. Abra um terminal na pasta do projeto
2. Execute: `fastapi dev prova_agent.py`
3. Aguarde o servidor iniciar em http://localhost:8000

In [ ]:
%pip install requests -q

In [ ]:
import requests
import json
from typing import Dict, Any, List
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURACAO - AgentOS
# ═══════════════════════════════════════════════════════════════════════════
AGENTOS_URL = "http://localhost:8000"

# ═══════════════════════════════════════════════════════════════════════════
# DADOS DO ALUNO
# ═══════════════════════════════════════════════════════════════════════════
NOME_ALUNO = ""  # <- Preencha seu nome aqui
DATA_PROVA = datetime.now().strftime("%Y-%m-%d %H:%M")

# Verifica conexao com AgentOS
try:
    health = requests.get(f"{AGENTOS_URL}/api/health", timeout=5)
    if health.status_code == 200:
        info = health.json()
        print("[OK] Conectado ao AgentOS!")
        print(f"Modelo: {info.get('model', 'N/A')}")
        print(f"Agentes: {', '.join(info.get('agents', []))}")
    else:
        print("[AVISO] AgentOS respondeu com status:", health.status_code)
except requests.exceptions.ConnectionError:
    print("[ERRO] Nao foi possivel conectar ao AgentOS!")
    print("Execute 'fastapi dev prova_agent.py' em outro terminal")
except Exception as e:
    print(f"[ERRO] {e}")

print(f"\nData da prova: {DATA_PROVA}")
print(f"Aluno: {NOME_ALUNO or '(preencha seu nome)'}")

Data da prova: 2026-03-10 07:23
Modelo: llama-3.3-70b-versatile (via Groq)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FUNCAO DE AVALIACAO (via AgentOS API)
# ═══════════════════════════════════════════════════════════════════════════

# Armazena resultados
resultados_prova: Dict[str, Any] = {
    "aluno": NOME_ALUNO,
    "data": DATA_PROVA,
    "questoes": {},
    "nota_total": 0,
}

def avaliar_questao(numero: int, pergunta: str, resposta_aluno: str, 
                    criterios: str, peso: float = 1.0) -> Dict[str, Any]:
    """
    Avalia uma questao usando o AgentOS API.
    
    Args:
        numero: Numero da questao
        pergunta: Texto da pergunta
        resposta_aluno: Resposta fornecida pelo aluno
        criterios: Criterios de avaliacao e resposta esperada
        peso: Peso da questao na nota final (default 1.0)
    """
    try:
        # Chama API do AgentOS
        response = requests.post(
            f"{AGENTOS_URL}/api/avaliar",
            json={
                "numero_questao": numero,
                "pergunta": pergunta,
                "resposta_aluno": resposta_aluno,
                "criterios": criterios
            },
            timeout=60
        )
        
        if response.status_code != 200:
            raise Exception(f"API retornou status {response.status_code}")
        
        # Extrai resultado
        resultado = response.json().get("resultado", "")
        
        # Limpa e faz parse do JSON
        texto_limpo = resultado.strip()
        if texto_limpo.startswith("```"):
            texto_limpo = texto_limpo.split("```")[1]
            if texto_limpo.startswith("json"):
                texto_limpo = texto_limpo[4:]
        
        avaliacao = json.loads(texto_limpo)
        
    except requests.exceptions.ConnectionError:
        avaliacao = {
            "nota": 0,
            "status": "erro",
            "feedback": "Nao foi possivel conectar ao AgentOS. Verifique se o servidor esta rodando.",
            "dica": "Execute: fastapi dev prova_agent.py"
        }
    except json.JSONDecodeError as e:
        avaliacao = {
            "nota": 5,
            "status": "erro",
            "feedback": f"Erro ao processar resposta do agente: {str(e)}",
            "dica": "Tente novamente"
        }
    except Exception as e:
        avaliacao = {
            "nota": 5,
            "status": "erro",
            "feedback": f"Erro: {str(e)}",
            "dica": "Verifique sua conexao com o AgentOS"
        }
    
    # Armazena resultado
    avaliacao["peso"] = peso
    avaliacao["nota_ponderada"] = avaliacao.get("nota", 0) * peso
    resultados_prova["questoes"][f"Q{numero}"] = {
        "pergunta": pergunta,
        "resposta": resposta_aluno,
        "avaliacao": avaliacao
    }
    
    # Mostra feedback
    status_emoji = {"correta": "[OK]", "parcial": "[PARCIAL]", "incorreta": "[X]", "erro": "[!]"}
    print(f"\n{'='*60}")
    print(f"QUESTAO {numero} - {status_emoji.get(avaliacao.get('status', '?'), '[?]')} Nota: {avaliacao.get('nota', 0)}/10")
    print(f"{'='*60}")
    print(f"Feedback: {avaliacao.get('feedback', 'Sem feedback')}")
    if avaliacao.get('dica'):
        print(f"Dica: {avaliacao['dica']}")
    
    return avaliacao

def perguntar_tutor(pergunta: str) -> str:
    """
    Faz uma pergunta ao Tutor IA (para tirar duvidas).
    O tutor NAO da respostas diretas da prova.
    """
    try:
        # Usa o endpoint padrao do AgentOS para o Tutor
        response = requests.post(
            f"{AGENTOS_URL}/v1/runs",
            json={
                "agent_id": "tutor-ia",
                "message": pergunta
            },
            timeout=60
        )
        
        if response.status_code == 200:
            return response.json().get("content", "Sem resposta")
        else:
            return f"Erro: {response.status_code}"
    except Exception as e:
        return f"Erro ao conectar ao tutor: {e}"

print("[OK] Funcoes de avaliacao configuradas!")

[OK] Agente avaliador configurado!


---
## Parte 1 - Tokens (2 questoes)

### Questao 1 (Teorica - Peso 1.0)

**O que sao tokens em modelos de linguagem e por que eles sao importantes para o custo de uso de APIs de IA?**

In [ ]:
# RESPOSTA DO ALUNO - Questao 1
resposta_q1 = """

ESCREVA SUA RESPOSTA AQUI

"""

In [ ]:
# AVALIACAO - Questao 1
avaliar_questao(
    numero=1,
    pergunta="O que sao tokens em modelos de linguagem e por que eles sao importantes para o custo de uso de APIs de IA?",
    resposta_aluno=resposta_q1,
    criterios="""
    Resposta esperada deve mencionar:
    - Tokens sao unidades de texto (pedacos de palavras, palavras ou caracteres)
    - O modelo processa texto token por token
    - APIs cobram por numero de tokens (entrada + saida)
    - Diferentes idiomas usam quantidades diferentes de tokens
    Pontuacao:
    - 10: Menciona todos os pontos com clareza
    - 7-9: Menciona a maioria dos pontos
    - 4-6: Menciona alguns pontos mas falta profundidade
    - 1-3: Resposta vaga ou incompleta
    - 0: Nao respondeu ou resposta errada
    """,
    peso=1.0
)

### Questao 2 (Pratica - Peso 1.5)

**Dado que o Claude cobra $3 por milhao de tokens de entrada e $15 por milhao de tokens de saida, quanto custaria uma chamada com 500 tokens de entrada e 200 tokens de saida?**

Mostre o calculo completo.

In [ ]:
# RESPOSTA DO ALUNO - Questao 2
resposta_q2 = """

ESCREVA SUA RESPOSTA AQUI (inclua o calculo)

"""

In [ ]:
# AVALIACAO - Questao 2
avaliar_questao(
    numero=2,
    pergunta="Calcule o custo de 500 tokens entrada + 200 tokens saida (precos Claude)",
    resposta_aluno=resposta_q2,
    criterios="""
    Calculo correto:
    - Entrada: 500 * ($3 / 1.000.000) = $0.0015
    - Saida: 200 * ($15 / 1.000.000) = $0.003
    - Total: $0.0045 (ou aproximadamente meio centavo de dolar)
    Pontuacao:
    - 10: Calculo completo e correto com explicacao
    - 7-9: Resultado correto mas explicacao incompleta
    - 4-6: Metodologia correta mas erro de calculo
    - 1-3: Tentou mas errou a logica
    - 0: Nao respondeu
    """,
    peso=1.5
)

---
## Parte 2 - Prompt Engineering (2 questoes)

### Questao 3 (Teorica - Peso 1.0)

**Liste e explique brevemente 3 tecnicas de Prompt Engineering que voce aprendeu.**

In [ ]:
# RESPOSTA DO ALUNO - Questao 3
resposta_q3 = """

ESCREVA SUA RESPOSTA AQUI

"""

In [ ]:
# AVALIACAO - Questao 3
avaliar_questao(
    numero=3,
    pergunta="Liste e explique 3 tecnicas de Prompt Engineering",
    resposta_aluno=resposta_q3,
    criterios="""
    Tecnicas validas (precisa de 3):
    - Persona: definir um papel para o modelo assumir
    - Chain-of-Thought: pedir raciocinio passo a passo
    - Few-Shot: dar exemplos do formato desejado
    - Restricoes: delimitar formato, tom, tamanho
    - XML Tags: estruturar prompts complexos
    Pontuacao:
    - 10: 3 tecnicas corretas com boas explicacoes
    - 7-9: 3 tecnicas mas explicacoes superficiais
    - 4-6: 2 tecnicas corretas ou 3 com explicacoes fracas
    - 1-3: Apenas 1 tecnica correta
    - 0: Nenhuma tecnica valida
    """,
    peso=1.0
)

### Questao 4 (Pratica - Peso 2.0)

**Reescreva o prompt fraco abaixo usando pelo menos 2 tecnicas de Prompt Engineering:**

Prompt fraco: `"Me explica fisica"`

Crie um prompt forte que gere uma resposta melhor.

In [ ]:
# RESPOSTA DO ALUNO - Questao 4
resposta_q4 = """

ESCREVA SEU PROMPT FORTE AQUI

"""

In [ ]:
# AVALIACAO - Questao 4
avaliar_questao(
    numero=4,
    pergunta="Reescreva 'Me explica fisica' usando 2+ tecnicas de Prompt Engineering",
    resposta_aluno=resposta_q4,
    criterios="""
    O prompt deve:
    - Usar pelo menos 2 tecnicas visiveis (Persona, CoT, Few-Shot, Restricoes, XML)
    - Ser especifico sobre o topico de fisica
    - Definir publico-alvo ou nivel
    - Ter restricoes de formato ou tamanho
    Exemplo bom: "Voce e um professor de fisica do ensino medio. Explique a Lei da Gravidade
    para um aluno de 15 anos usando uma analogia do cotidiano. Estruture em: 1) O que e,
    2) Como funciona, 3) Exemplo pratico. Maximo 100 palavras."
    Pontuacao:
    - 10: 2+ tecnicas claras, prompt bem estruturado
    - 7-9: 2 tecnicas mas poderia ser melhor
    - 4-6: 1 tecnica visivel ou prompt generico
    - 1-3: Pouca melhoria em relacao ao original
    - 0: Nao melhorou ou piorou
    """,
    peso=2.0
)

---
## Parte 3 - Temperatura (2 questoes)

### Questao 5 (Teorica - Peso 1.0)

**O que o parametro "temperatura" controla em modelos de linguagem? Qual a diferenca entre temperatura 0 e temperatura alta?**

In [ ]:
# RESPOSTA DO ALUNO - Questao 5
resposta_q5 = """

ESCREVA SUA RESPOSTA AQUI

"""

In [ ]:
# AVALIACAO - Questao 5
avaliar_questao(
    numero=5,
    pergunta="O que o parametro temperatura controla? Diferenca entre temp 0 e alta?",
    resposta_aluno=resposta_q5,
    criterios="""
    Resposta esperada:
    - Temperatura controla aleatoriedade/criatividade na escolha de palavras
    - Temperatura 0: deterministico, sempre escolhe palavra mais provavel, previsivel
    - Temperatura alta: mais aleatorio, aceita palavras menos provaveis, criativo
    - Pode mencionar: temp 0 para codigo/factuais, temp alta para criatividade
    Pontuacao:
    - 10: Explica bem ambos os extremos com exemplos de uso
    - 7-9: Entende o conceito mas falta profundidade
    - 4-6: Entendimento parcial
    - 1-3: Confuso ou muito vago
    - 0: Errado ou nao respondeu
    """,
    peso=1.0
)

### Questao 6 (Aplicacao - Peso 1.5)

**Para cada tarefa abaixo, indique se voce usaria temperatura BAIXA (0-0.3), MEDIA (0.5-0.7) ou ALTA (1.0+). Justifique brevemente cada escolha.**

a) Gerar codigo Python  
b) Escrever um poema  
c) Resumir um documento  

In [ ]:
# RESPOSTA DO ALUNO - Questao 6
resposta_q6 = """
a) Gerar codigo Python: 

b) Escrever um poema: 

c) Resumir um documento: 

"""

In [ ]:
# AVALIACAO - Questao 6
avaliar_questao(
    numero=6,
    pergunta="Qual temperatura para: a) codigo, b) poema, c) resumo?",
    resposta_aluno=resposta_q6,
    criterios="""
    Respostas esperadas:
    a) Codigo Python: BAIXA (0-0.3) - precisa ser preciso, correto, deterministico
    b) Poema: ALTA (1.0+) - criatividade, originalidade, variedade
    c) Resumo: BAIXA a MEDIA (0-0.5) - fidelidade ao texto, mas alguma flexibilidade
    Pontuacao:
    - 10: Todas corretas com boas justificativas
    - 7-9: Todas corretas mas justificativas fracas
    - 4-6: 2 de 3 corretas
    - 1-3: 1 de 3 correta
    - 0: Todas erradas ou nao respondeu
    """,
    peso=1.5
)

---
## Parte 4 - Alucinacoes (2 questoes)

### Questao 7 (Teorica - Peso 1.0)

**O que sao "alucinacoes" em modelos de IA? Por que elas acontecem?**

In [ ]:
# RESPOSTA DO ALUNO - Questao 7
resposta_q7 = """

ESCREVA SUA RESPOSTA AQUI

"""

In [ ]:
# AVALIACAO - Questao 7
avaliar_questao(
    numero=7,
    pergunta="O que sao alucinacoes em IA? Por que acontecem?",
    resposta_aluno=resposta_q7,
    criterios="""
    Resposta esperada:
    - Alucinacoes: quando a IA gera informacoes falsas mas plausíveis
    - A IA gera texto estatisticamente provavel, nao verificado
    - Ela preve proxima palavra, nao consulta banco de fatos
    - Comum em: citacoes, pessoas especificas, dados recentes, DOIs
    Pontuacao:
    - 10: Definicao clara + explica mecanismo
    - 7-9: Entende o conceito mas explicacao superficial
    - 4-6: Definicao correta mas nao explica por que acontece
    - 1-3: Entendimento vago
    - 0: Errado ou nao respondeu
    """,
    peso=1.0
)

### Questao 8 (Pratica - Peso 1.5)

**Cite 2 estrategias para reduzir alucinacoes em sistemas de IA. Explique brevemente como cada uma funciona.**

In [ ]:
# RESPOSTA DO ALUNO - Questao 8
resposta_q8 = """

ESCREVA SUA RESPOSTA AQUI

"""

In [ ]:
# AVALIACAO - Questao 8
avaliar_questao(
    numero=8,
    pergunta="Cite 2 estrategias para reduzir alucinacoes",
    resposta_aluno=resposta_q8,
    criterios="""
    Estrategias validas:
    - RAG (Retrieval-Augmented Generation): alimentar modelo com dados verificados
    - Grounding: conectar a fontes externas confiaveis
    - Self-consistency: pedir ao modelo verificar proprias respostas
    - Confianca calibrada: pedir niveis de certeza
    - Verificacao humana: revisar respostas criticas
    Pontuacao:
    - 10: 2 estrategias validas com boas explicacoes
    - 7-9: 2 estrategias mas explicacoes fracas
    - 4-6: 1 estrategia bem explicada
    - 1-3: Ideias vagas ou parcialmente corretas
    - 0: Nenhuma estrategia valida
    """,
    peso=1.5
)

---
## Parte 5 - Etica e Vies (2 questoes)

### Questao 9 (Teorica - Peso 1.0)

**De onde vem o vies em modelos de IA? A IA "cria" preconceitos ou os reproduz? Explique.**

In [ ]:
# RESPOSTA DO ALUNO - Questao 9
resposta_q9 = """

ESCREVA SUA RESPOSTA AQUI

"""

In [ ]:
# AVALIACAO - Questao 9
avaliar_questao(
    numero=9,
    pergunta="De onde vem o vies em IA? A IA cria ou reproduz preconceitos?",
    resposta_aluno=resposta_q9,
    criterios="""
    Resposta esperada:
    - A IA aprende com dados da internet que contem preconceitos historicos
    - Ela REPRODUZ e AMPLIFICA padroes existentes, nao cria do nada
    - Exemplos: vies de genero em profissoes, vies geografico, racial
    - Os dados refletem desigualdades sociais historicas
    Pontuacao:
    - 10: Entende que reproduz + explica origem nos dados + exemplos
    - 7-9: Conceito correto mas falta profundidade
    - 4-6: Entendimento parcial
    - 1-3: Confuso ou muito vago
    - 0: Acha que IA cria preconceitos ou nao respondeu
    """,
    peso=1.0
)

### Questao 10 (Reflexao - Peso 2.0)

**Uma empresa usa IA para selecionar curriculos automaticamente. Descobrem que o sistema da notas menores para candidatas mulheres. De quem e a responsabilidade? Distribua entre: programador, empresa, dados historicos e sociedade. Justifique.**

In [ ]:
# RESPOSTA DO ALUNO - Questao 10
resposta_q10 = """

ESCREVA SUA RESPOSTA AQUI (minimo 5 linhas)

"""

In [ ]:
# AVALIACAO - Questao 10
avaliar_questao(
    numero=10,
    pergunta="Quem e responsavel pelo vies em IA de selecao de curriculos?",
    resposta_aluno=resposta_q10,
    criterios="""
    Avalie a qualidade da reflexao etica:
    - Considera multiplos atores (programador, empresa, dados, sociedade)?
    - Distribui responsabilidade de forma equilibrada?
    - Reconhece que nao ha um unico culpado?
    - Propoe acoes concretas para cada ator?
    - Demonstra pensamento critico?
    Pontuacao:
    - 10: Reflexao profunda, equilibrada, com nuances eticas
    - 7-9: Boa reflexao mas falta algum elemento
    - 4-6: Reflexao superficial ou culpa apenas um ator
    - 1-3: Muito curta ou sem profundidade
    - 0: Nao respondeu ou resposta irrelevante
    """,
    peso=2.0
)

---
## Resultado Final

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CALCULO DA NOTA FINAL
# ═══════════════════════════════════════════════════════════════════════════

def calcular_resultado_final():
    """Calcula e exibe o resultado final da prova."""
    
    questoes = resultados_prova["questoes"]
    
    if not questoes:
        print("[AVISO] Nenhuma questao foi avaliada ainda!")
        print("Execute as celulas de avaliacao primeiro.")
        return
    
    # Calcula totais
    soma_notas_ponderadas = 0
    soma_pesos = 0
    
    print("\n" + "="*70)
    print("                    RESULTADO FINAL DA PROVA")
    print("="*70)
    print(f"Aluno: {resultados_prova['aluno'] or '(nao informado)'}")
    print(f"Data: {resultados_prova['data']}")
    print("-"*70)
    print(f"{'Questao':<12} {'Nota':<8} {'Peso':<8} {'Ponderada':<12} {'Status'}")
    print("-"*70)
    
    for q_id, q_data in sorted(questoes.items()):
        aval = q_data["avaliacao"]
        nota = aval.get("nota", 0)
        peso = aval.get("peso", 1.0)
        ponderada = nota * peso
        status = aval.get("status", "?")
        
        soma_notas_ponderadas += ponderada
        soma_pesos += peso
        
        status_display = {
            "correta": "Correta",
            "parcial": "Parcial",
            "incorreta": "Incorreta",
            "erro": "Erro"
        }.get(status, status)
        
        print(f"{q_id:<12} {nota:<8} {peso:<8} {ponderada:<12.1f} {status_display}")
    
    # Nota final (media ponderada convertida para escala 0-10)
    if soma_pesos > 0:
        nota_final = soma_notas_ponderadas / soma_pesos
    else:
        nota_final = 0
    
    resultados_prova["nota_total"] = nota_final
    
    print("-"*70)
    print(f"{'TOTAL':<12} {'':<8} {soma_pesos:<8} {soma_notas_ponderadas:<12.1f}")
    print("="*70)
    print(f"\n                    NOTA FINAL: {nota_final:.1f} / 10")
    print("="*70)
    
    # Conceito
    if nota_final >= 9:
        conceito = "A - Excelente!"
    elif nota_final >= 7:
        conceito = "B - Muito Bom"
    elif nota_final >= 5:
        conceito = "C - Aprovado"
    elif nota_final >= 3:
        conceito = "D - Recuperacao"
    else:
        conceito = "E - Reprovado"
    
    print(f"                    Conceito: {conceito}")
    print("="*70)
    
    # Salva resultados em arquivo
    nome_arquivo = f"resultado_prova_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(nome_arquivo, 'w', encoding='utf-8') as f:
        json.dump(resultados_prova, f, ensure_ascii=False, indent=2)
    print(f"\nResultado salvo em: {nome_arquivo}")

# Executa
calcular_resultado_final()

---

## Parabens por completar a prova!

**Proximos passos:**
- Revise as questoes onde tirou nota menor
- Releia as atividades correspondentes
- Discuta com seus colegas as questoes de etica

---

*Prova criada com Agno AI + Groq (LLama 3.3 70B)*  
*TransformAI - Global Shapers Hub Porto Alegre*